<a href="https://colab.research.google.com/github/MukeshReddy8977/AI-COURSE-PROGRAMS-2026/blob/main/FastAPI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from fastapi import FastAPI, HTTPException, Query, Path
from fastapi import Response, Header, Depends
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field, ConfigDict
from typing import Annotated

app = FastAPI(title='My Task API', version='1.0.0') # app created

@app.get('/')  #routing
def home():
    return {'message': 'My Task API is working'}

@app.get('/index')  #routing
def index():
    return {'message': 'this is Mukesh application'}

client = TestClient(app)
response = client.get('/index')
print(response.status_code)
print(response.json())

200
{'message': 'this is Mukesh application'}


In [5]:
@app.get('/numbers/{number}')
def describe_number(number: Annotated[int, Path(ge=1)]):
    return {'number': number, 'intothree': number * 3}

good = client.get('/numbers/7')
print(good.status_code, good.json())
print(client.get('/numbers/abc').status_code)
print(client.get('/numbers/0').status_code)

200 {'number': 7, 'intothree': 21}
422
422


In [8]:
#parameters
@app.get('/greet')
def greeting(name: Annotated[str, Query(min_length=1)] = 'friend', loud: bool = False):
    message = 'Hello, ' + name
    if loud:
        message = message.upper()
    return {'message': message}

print(client.get('/greet').json())
r = client.get('/greet', params={'name': 'Mukesh', 'loud': False})
print(r.json())
print(client.get('/greet', params={'name': ''}).status_code)

{'message': 'Hello, friend'}
{'message': 'Hello, Mukesh'}
422


In [10]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI()  # Fresh app: clears the old routes

@app.get('/greet')
def greeting(name: str = 'friend', loud: bool = False):
    message = 'Hello, ' + name
    if loud:
        message = message.upper()
    return {'message': message}

client = TestClient(app)  # Client for the fresh app
response = client.get('/greet', params={'name': ''})
print(response.status_code)
print(response.json())

200
{'message': 'Hello, '}


In [12]:
@app.get('/greet')
def greeting(name: str = 'friend', loud: bool = False):
    message = 'Hello, ' + name
    if loud:
        message = message.upper()
    return {'message': message}

print(client.get('/greet').json())
r = client.get('/greet', params={'name': 'Asha', 'loud': True})
print(r.json())
print(client.get('/greet', params={'name': ''}).status_code)

{'message': 'Hello, friend'}
{'message': 'HELLO, ASHA'}
200


In [14]:
class TaskCreate(BaseModel):
    model_config = ConfigDict(extra='forbid')
    title: str = Field(min_length=1, max_length=100)
    done: bool = False

class TaskOut(BaseModel):
    id: int
    title: str
    done: bool

sample = TaskCreate(title='Read the guide')
print(sample.model_dump())

{'title': 'Read the guide', 'done': False}


In [16]:
tasks: dict[int, dict] = {}
next_id = 1

@app.post('/tasks', response_model=TaskOut, status_code=201)
def create_task(task: TaskCreate):
    global next_id
    saved = {'id': next_id, **task.model_dump()}
    tasks[next_id] = saved
    next_id += 1
    return saved

created = client.post('/tasks', json={'title': 'my first save post call for id 1'})
print(created.status_code, created.json())
task_id = created.json()['id']

201 {'id': 1, 'title': 'my first save post call for id 1', 'done': False}
